In [0]:
storage_account_name = ""
storage_account_key = ""
spark.conf.set(
f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
storage_account_key
)

In [0]:
# Load silver layer
train_silver_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/FD001/train_silver"
test_silver_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/FD001/test_silver"

train_df = spark.read.parquet(train_silver_path)
test_df = spark.read.parquet(test_silver_path)

print("=== SILVER DATA LOADED ===")
print(f"Train rows: {train_df.count()}")
print(f"Test rows: {test_df.count()}")
print(f"Columns: {train_df.columns}")
display(train_df.limit(5))

=== SILVER DATA LOADED ===
Train rows: 20631
Test rows: 13096
Columns: ['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21
1,1,-7.0E-4,-4.0E-4,100.0,641.82,1589.7,1400.6,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392.0,39.06,23.419
1,2,0.0019,-3.0E-4,100.0,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392.0,39.0,23.4236
1,3,-0.0043,3.0E-4,100.0,642.35,1587.99,1404.2,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390.0,38.95,23.3442
1,4,7.0E-4,0.0,100.0,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392.0,38.88,23.3739
1,5,-0.0019,-2.0E-4,100.0,642.37,1582.85,1406.22,554.0,2388.06,9055.15,47.28,522.19,2388.04,8133.8,8.4294,393.0,38.9,23.4044


In [0]:
from pyspark.sql.functions import max as spark_max, when, col
from pyspark.sql import Window

# Compute max cycle per engine
window = Window.partitionBy("engine_id")
train_df = train_df.withColumn("max_cycle", spark_max("cycle").over(window))

# RUL = max_cycle - current_cycle
train_df = train_df.withColumn("RUL", col("max_cycle") - col("cycle"))
train_df = train_df.drop("max_cycle")

print("=== RUL COMPUTED ===")
print("Sample RUL values:")
display(train_df.select("engine_id", "cycle", "RUL").limit(10))

=== RUL COMPUTED ===
Sample RUL values:


engine_id,cycle,RUL
1,1,191
1,2,190
1,3,189
1,4,188
1,5,187
1,6,186
1,7,185
1,8,184
1,9,183
1,10,182


In [0]:
# Piecewise linear degradation assumption
# Engines are assumed healthy until 125 cycles before failure
RUL_CLIP = 125

train_df = train_df.withColumn("RUL",
    when(col("RUL") > RUL_CLIP, RUL_CLIP).otherwise(col("RUL"))
)

print(f"RUL clipped at {RUL_CLIP}")
print("\n=== RUL DISTRIBUTION AFTER CLIPPING ===")
train_df.describe("RUL").show()
display(train_df.select("engine_id", "cycle", "RUL").limit(10))

RUL clipped at 125

=== RUL DISTRIBUTION AFTER CLIPPING ===
+-------+-----------------+
|summary|              RUL|
+-------+-----------------+
|  count|            20631|
|   mean|86.82928602588338|
| stddev|41.67369885453269|
|    min|                0|
|    max|              125|
+-------+-----------------+



engine_id,cycle,RUL
1,1,125
1,2,125
1,3,125
1,4,125
1,5,125
1,6,125
1,7,125
1,8,125
1,9,125
1,10,125


In [0]:
from pyspark.ml.feature import MinMaxScaler, VectorAssembler
from pyspark.ml import Pipeline

# Columns to normalize
feature_cols = [c for c in train_df.columns 
                if c.startswith("sensor") or c.startswith("op_setting")]

print(f"Columns to normalize: {feature_cols}")

# Step 1: Assemble into a vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_vec"
)

# Step 2: Scale to [0,1]
scaler = MinMaxScaler(
    inputCol="features_vec",
    outputCol="scaled_vec"
)

# Step 3: Fit on training data only
pipeline = Pipeline(stages=[assembler, scaler])
scaler_model = pipeline.fit(train_df)

# Step 4: Transform both train and test
train_scaled = scaler_model.transform(train_df)
test_scaled = scaler_model.transform(test_df)

print("Normalization complete")
display(train_scaled.select("engine_id", "cycle", "RUL", "scaled_vec").limit(5))

Columns to normalize: ['op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']
Normalization complete


engine_id,cycle,RUL,scaled_vec
1,1,125,"Map(vectorType -> dense, length -> 17, values -> List(0.45977011694199915, 0.16666669091985756, 0.5, 0.1837301222538836, 0.4067999584849122, 0.3097565599409842, 0.7262469900240798, 0.24260355029585798, 0.10975513566570848, 0.3690491868792603, 0.6332556838146302, 0.20603015075376885, 0.19960893108670288, 0.36398862489679845, 0.3333333333333333, 0.7131793261297706, 0.7246622244462588))"
1,2,125,"Map(vectorType -> dense, length -> 17, values -> List(0.6091954026064614, 0.25, 0.5, 0.28313264086772677, 0.45301742835320824, 0.35263366124452394, 0.6280210329745933, 0.21227810650887574, 0.10024188459650844, 0.3809542190998224, 0.765463749821059, 0.27961234745154345, 0.16281519222095392, 0.41131272360333915, 0.3333333333333333, 0.6666666666666667, 0.7310121414838421))"
1,3,125,"Map(vectorType -> dense, length -> 17, values -> List(0.25287355829661745, 0.75, 0.5, 0.34335876459233383, 0.3695215386130235, 0.3705259080062807, 0.7101479188166495, 0.2729289940828402, 0.14004329383720115, 0.2500011353263609, 0.7953045899975273, 0.22074659009332376, 0.17179314671887638, 0.35744610586184755, 0.16666666666666666, 0.6279075956778624, 0.6213753325080987))"
1,4,125,"Map(vectorType -> dense, length -> 17, values -> List(0.5402298830580008, 0.5, 0.5, 0.34335876459233383, 0.25615873666394334, 0.3311951105927541, 0.7407440169050077, 0.3184171597633136, 0.12451798359391432, 0.1666681804351479, 0.889121692846267, 0.29432878679109836, 0.17488994887378703, 0.16660489863315292, 0.3333333333333333, 0.5736443048680234, 0.6623851036371777))"
1,5,125,"Map(vectorType -> dense, length -> 17, values -> List(0.39080459739353857, 0.33333334545992876, 0.5, 0.34938873058185493, 0.2574653974404352, 0.404624825363594, 0.6682785394859698, 0.24260355029585798, 0.14995968590058192, 0.255952516110281, 0.7462682682422144, 0.2354630294328787, 0.17473372289038228, 0.4020805430694432, 0.41666666666666663, 0.5891485246890581, 0.7045010403223683))"


In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col as scol

# Convert scaled vector back to individual columns
train_scaled = train_scaled.withColumn("scaled_array", vector_to_array("scaled_vec"))
test_scaled = test_scaled.withColumn("scaled_array", vector_to_array("scaled_vec"))

# Create new scaled column names
scaled_cols = [f"{c}_scaled" for c in feature_cols]

# Unpack array into individual columns
for i, c in enumerate(scaled_cols):
    train_scaled = train_scaled.withColumn(c, scol("scaled_array")[i])
    test_scaled = test_scaled.withColumn(c, scol("scaled_array")[i])

# Keep only relevant columns
cols_to_keep_train = ["engine_id", "cycle", "RUL"] + scaled_cols
cols_to_keep_test = ["engine_id", "cycle"] + scaled_cols

train_final = train_scaled.select(cols_to_keep_train)
test_final = test_scaled.select(cols_to_keep_test)

print("Scaled columns unpacked")
print(f"Train columns: {train_final.columns}")
display(train_final.limit(5))

Scaled columns unpacked
Train columns: ['engine_id', 'cycle', 'RUL', 'op_setting_1_scaled', 'op_setting_2_scaled', 'op_setting_3_scaled', 'sensor_2_scaled', 'sensor_3_scaled', 'sensor_4_scaled', 'sensor_7_scaled', 'sensor_8_scaled', 'sensor_9_scaled', 'sensor_11_scaled', 'sensor_12_scaled', 'sensor_13_scaled', 'sensor_14_scaled', 'sensor_15_scaled', 'sensor_17_scaled', 'sensor_20_scaled', 'sensor_21_scaled']


engine_id,cycle,RUL,op_setting_1_scaled,op_setting_2_scaled,op_setting_3_scaled,sensor_2_scaled,sensor_3_scaled,sensor_4_scaled,sensor_7_scaled,sensor_8_scaled,sensor_9_scaled,sensor_11_scaled,sensor_12_scaled,sensor_13_scaled,sensor_14_scaled,sensor_15_scaled,sensor_17_scaled,sensor_20_scaled,sensor_21_scaled
1,1,125,0.45977011694199915,0.16666669091985756,0.5,0.1837301222538836,0.4067999584849122,0.3097565599409842,0.7262469900240798,0.24260355029585798,0.10975513566570848,0.3690491868792603,0.6332556838146302,0.20603015075376885,0.19960893108670288,0.36398862489679845,0.3333333333333333,0.7131793261297706,0.7246622244462588
1,2,125,0.6091954026064614,0.25,0.5,0.28313264086772677,0.45301742835320824,0.35263366124452394,0.6280210329745933,0.21227810650887574,0.10024188459650844,0.3809542190998224,0.765463749821059,0.27961234745154345,0.16281519222095392,0.41131272360333915,0.3333333333333333,0.6666666666666667,0.7310121414838421
1,3,125,0.25287355829661745,0.75,0.5,0.34335876459233383,0.3695215386130235,0.3705259080062807,0.7101479188166495,0.2729289940828402,0.14004329383720115,0.2500011353263609,0.7953045899975273,0.22074659009332376,0.17179314671887638,0.35744610586184755,0.16666666666666666,0.6279075956778624,0.6213753325080987
1,4,125,0.5402298830580008,0.5,0.5,0.34335876459233383,0.25615873666394334,0.3311951105927541,0.7407440169050077,0.3184171597633136,0.12451798359391432,0.1666681804351479,0.889121692846267,0.29432878679109836,0.17488994887378703,0.16660489863315292,0.3333333333333333,0.5736443048680234,0.6623851036371777
1,5,125,0.39080459739353857,0.33333334545992876,0.5,0.34938873058185493,0.2574653974404352,0.404624825363594,0.6682785394859698,0.24260355029585798,0.14995968590058192,0.255952516110281,0.7462682682422144,0.2354630294328787,0.17473372289038228,0.4020805430694432,0.41666666666666663,0.5891485246890581,0.7045010403223683


In [0]:
# Write to curated container (Gold layer)
train_gold_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/FD001/train_gold"
test_gold_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/FD001/test_gold"

train_final.write.mode("overwrite").parquet(train_gold_path)
test_final.write.mode("overwrite").parquet(test_gold_path)

print("Gold layer written successfully")
print(f"Train gold: {train_gold_path}")
print(f"Test gold: {test_gold_path}")

Gold layer written successfully
Train gold: abfss://curated@datalake60306249.dfs.core.windows.net/FD001/train_gold
Test gold: abfss://curated@datalake60306249.dfs.core.windows.net/FD001/test_gold


In [0]:
# Verify gold layer
train_check = spark.read.parquet(train_gold_path)
test_check = spark.read.parquet(test_gold_path)

print("=== GOLD LAYER VERIFICATION ===")
print(f"Train rows: {train_check.count()}")
print(f"Test rows: {test_check.count()}")
print(f"Columns: {len(train_check.columns)}")
train_check.printSchema()
display(train_check.limit(5))

=== GOLD LAYER VERIFICATION ===
Train rows: 20631
Test rows: 13096
Columns: 20
root
 |-- engine_id: integer (nullable = true)
 |-- cycle: integer (nullable = true)
 |-- RUL: integer (nullable = true)
 |-- op_setting_1_scaled: double (nullable = true)
 |-- op_setting_2_scaled: double (nullable = true)
 |-- op_setting_3_scaled: double (nullable = true)
 |-- sensor_2_scaled: double (nullable = true)
 |-- sensor_3_scaled: double (nullable = true)
 |-- sensor_4_scaled: double (nullable = true)
 |-- sensor_7_scaled: double (nullable = true)
 |-- sensor_8_scaled: double (nullable = true)
 |-- sensor_9_scaled: double (nullable = true)
 |-- sensor_11_scaled: double (nullable = true)
 |-- sensor_12_scaled: double (nullable = true)
 |-- sensor_13_scaled: double (nullable = true)
 |-- sensor_14_scaled: double (nullable = true)
 |-- sensor_15_scaled: double (nullable = true)
 |-- sensor_17_scaled: double (nullable = true)
 |-- sensor_20_scaled: double (nullable = true)
 |-- sensor_21_scaled: double

engine_id,cycle,RUL,op_setting_1_scaled,op_setting_2_scaled,op_setting_3_scaled,sensor_2_scaled,sensor_3_scaled,sensor_4_scaled,sensor_7_scaled,sensor_8_scaled,sensor_9_scaled,sensor_11_scaled,sensor_12_scaled,sensor_13_scaled,sensor_14_scaled,sensor_15_scaled,sensor_17_scaled,sensor_20_scaled,sensor_21_scaled
1,1,125,0.45977011694199915,0.16666669091985756,0.5,0.1837301222538836,0.4067999584849122,0.3097565599409842,0.7262469900240798,0.24260355029585798,0.10975513566570848,0.3690491868792603,0.6332556838146302,0.20603015075376885,0.19960893108670288,0.36398862489679845,0.3333333333333333,0.7131793261297706,0.7246622244462588
1,2,125,0.6091954026064614,0.25,0.5,0.28313264086772677,0.45301742835320824,0.35263366124452394,0.6280210329745933,0.21227810650887574,0.10024188459650844,0.3809542190998224,0.765463749821059,0.27961234745154345,0.16281519222095392,0.41131272360333915,0.3333333333333333,0.6666666666666667,0.7310121414838421
1,3,125,0.25287355829661745,0.75,0.5,0.34335876459233383,0.3695215386130235,0.3705259080062807,0.7101479188166495,0.2729289940828402,0.14004329383720115,0.2500011353263609,0.7953045899975273,0.22074659009332376,0.17179314671887638,0.35744610586184755,0.16666666666666666,0.6279075956778624,0.6213753325080987
1,4,125,0.5402298830580008,0.5,0.5,0.34335876459233383,0.25615873666394334,0.3311951105927541,0.7407440169050077,0.3184171597633136,0.12451798359391432,0.1666681804351479,0.889121692846267,0.29432878679109836,0.17488994887378703,0.16660489863315292,0.3333333333333333,0.5736443048680234,0.6623851036371777
1,5,125,0.39080459739353857,0.33333334545992876,0.5,0.34938873058185493,0.2574653974404352,0.404624825363594,0.6682785394859698,0.24260355029585798,0.14995968590058192,0.255952516110281,0.7462682682422144,0.2354630294328787,0.17473372289038228,0.4020805430694432,0.41666666666666663,0.5891485246890581,0.7045010403223683


In [0]:
# Prepare tsfresh-ready format
# tsfresh needs: id column (engine_id), time column (cycle), + value columns

# Get scaled sensor and op_setting columns
feature_cols = [c for c in train_final.columns 
                if c.endswith("_scaled")]

print(f"Feature columns for tsfresh: {feature_cols}")
print(f"Total feature columns: {len(feature_cols)}")

# Build tsfresh-ready dataframes
tsfresh_cols = ["engine_id", "cycle"] + feature_cols

train_tsfresh = train_final.select(tsfresh_cols)
test_tsfresh = test_final.select(tsfresh_cols)

print("\ntsfresh-ready format confirmed")
print(f"Train shape: {train_tsfresh.count()} rows x {len(train_tsfresh.columns)} cols")
print(f"Test shape: {test_tsfresh.count()} rows x {len(test_tsfresh.columns)} cols")
display(train_tsfresh.limit(5))

Feature columns for tsfresh: ['op_setting_1_scaled', 'op_setting_2_scaled', 'op_setting_3_scaled', 'sensor_2_scaled', 'sensor_3_scaled', 'sensor_4_scaled', 'sensor_7_scaled', 'sensor_8_scaled', 'sensor_9_scaled', 'sensor_11_scaled', 'sensor_12_scaled', 'sensor_13_scaled', 'sensor_14_scaled', 'sensor_15_scaled', 'sensor_17_scaled', 'sensor_20_scaled', 'sensor_21_scaled']
Total feature columns: 17

tsfresh-ready format confirmed
Train shape: 20631 rows x 19 cols
Test shape: 13096 rows x 19 cols


engine_id,cycle,op_setting_1_scaled,op_setting_2_scaled,op_setting_3_scaled,sensor_2_scaled,sensor_3_scaled,sensor_4_scaled,sensor_7_scaled,sensor_8_scaled,sensor_9_scaled,sensor_11_scaled,sensor_12_scaled,sensor_13_scaled,sensor_14_scaled,sensor_15_scaled,sensor_17_scaled,sensor_20_scaled,sensor_21_scaled
1,1,0.45977011694199915,0.16666669091985756,0.5,0.1837301222538836,0.4067999584849122,0.3097565599409842,0.7262469900240798,0.24260355029585798,0.10975513566570848,0.3690491868792603,0.6332556838146302,0.20603015075376885,0.19960893108670288,0.36398862489679845,0.3333333333333333,0.7131793261297706,0.7246622244462588
1,2,0.6091954026064614,0.25,0.5,0.28313264086772677,0.45301742835320824,0.35263366124452394,0.6280210329745933,0.21227810650887574,0.10024188459650844,0.3809542190998224,0.765463749821059,0.27961234745154345,0.16281519222095392,0.41131272360333915,0.3333333333333333,0.6666666666666667,0.7310121414838421
1,3,0.25287355829661745,0.75,0.5,0.34335876459233383,0.3695215386130235,0.3705259080062807,0.7101479188166495,0.2729289940828402,0.14004329383720115,0.2500011353263609,0.7953045899975273,0.22074659009332376,0.17179314671887638,0.35744610586184755,0.16666666666666666,0.6279075956778624,0.6213753325080987
1,4,0.5402298830580008,0.5,0.5,0.34335876459233383,0.25615873666394334,0.3311951105927541,0.7407440169050077,0.3184171597633136,0.12451798359391432,0.1666681804351479,0.889121692846267,0.29432878679109836,0.17488994887378703,0.16660489863315292,0.3333333333333333,0.5736443048680234,0.6623851036371777
1,5,0.39080459739353857,0.33333334545992876,0.5,0.34938873058185493,0.2574653974404352,0.404624825363594,0.6682785394859698,0.24260355029585798,0.14995968590058192,0.255952516110281,0.7462682682422144,0.2354630294328787,0.17473372289038228,0.4020805430694432,0.41666666666666663,0.5891485246890581,0.7045010403223683


In [0]:
# Prepare RUL labels for each cycle
train_rul_labels = train_final.select("engine_id", "cycle", "RUL") \
    .withColumnRenamed("RUL", "target_RUL")

print("=== RUL LABELS PER ENGINE PER CYCLE ===")
print(f"Total rows: {train_rul_labels.count()}")
print(f"Total engines: {train_rul_labels.select('engine_id').distinct().count()}")
train_rul_labels.describe("target_RUL").show()
display(train_rul_labels.limit(15))

=== RUL LABELS PER ENGINE PER CYCLE ===
Total rows: 20631
Total engines: 100
+-------+-----------------+
|summary|       target_RUL|
+-------+-----------------+
|  count|            20631|
|   mean|86.82928602588338|
| stddev|41.67369885453269|
|    min|                0|
|    max|              125|
+-------+-----------------+



engine_id,cycle,target_RUL
1,1,125
1,2,125
1,3,125
1,4,125
1,5,125
1,6,125
1,7,125
1,8,125
1,9,125
1,10,125


In [0]:
# Write tsfresh-ready features and labels
train_features_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/FD001/train_tsfresh_ready"
test_features_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/FD001/test_tsfresh_ready"
train_labels_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/FD001/train_rul_labels"

train_tsfresh.write.mode("overwrite").parquet(train_features_path)
test_tsfresh.write.mode("overwrite").parquet(test_features_path)
train_rul_labels.write.mode("overwrite").parquet(train_labels_path)

print("tsfresh-ready data written to curated")
print(f"Train features: {train_features_path}")
print(f"Test features: {test_features_path}")
print(f"Train labels: {train_labels_path}")

tsfresh-ready data written to curated
Train features: abfss://curated@datalake60306249.dfs.core.windows.net/FD001/train_tsfresh_ready
Test features: abfss://curated@datalake60306249.dfs.core.windows.net/FD001/test_tsfresh_ready
Train labels: abfss://curated@datalake60306249.dfs.core.windows.net/FD001/train_rul_labels


In [0]:
# Final verification
train_feat_check = spark.read.parquet(train_features_path)
test_feat_check = spark.read.parquet(test_features_path)
labels_check = spark.read.parquet(train_labels_path)

print("=== FINAL VERIFICATION ===")
print(f"Train features rows: {train_feat_check.count()}")
print(f"Test features rows: {test_feat_check.count()}")
print(f"Train labels rows: {labels_check.count()}")
print(f"Feature columns: {len(train_feat_check.columns)}")
print("\n=== All data ready for Azure ML Pipeline ===")

=== FINAL VERIFICATION ===
Train features rows: 20631
Test features rows: 13096
Train labels rows: 20631
Feature columns: 19

=== All data ready for Azure ML Pipeline ===


In [0]:
# FINAL VERIFICATION
print("\n" + "="*60)
print("✅ NOTEBOOK 06 COMPLETE - TSFRESH-READY DATA")
print("="*60)
train_tsfresh_verify = spark.read.parquet(train_features_path)
test_tsfresh_verify = spark.read.parquet(test_features_path)
labels_verify = spark.read.parquet(train_labels_path)
print(f"✅ Train tsfresh-ready: {train_tsfresh_verify.count()} rows")
print(f"✅ Test tsfresh-ready: {test_tsfresh_verify.count()} rows")
print(f"✅ Train RUL labels: {labels_verify.count()} rows")
print(f"✅ Features path: {train_features_path}")
print("="*60)


✅ NOTEBOOK 06 COMPLETE - TSFRESH-READY DATA
✅ Train tsfresh-ready: 20631 rows
✅ Test tsfresh-ready: 13096 rows
✅ Train RUL labels: 20631 rows
✅ Features path: abfss://curated@datalake60306249.dfs.core.windows.net/FD001/train_tsfresh_ready
